# 08 Claude red-team: root replay, custody attacks, context ablation
Replays only PUBLIC artifacts. Model inference ran on magicSTUDIObox and MacBookPro via `redteam/ablation_runner.py`; this notebook recomputes every statistic from the public per-case bits. Hashes prove identity, not causality.

In [ ]:
import json, subprocess, sys
from pathlib import Path
R = Path.cwd()
sys.path.insert(0, str(R/'redteam'))
r = subprocess.run([sys.executable, 'redteam/verify_roots.py', '--repo', '.'], capture_output=True, text=True, check=True)
rep = json.loads(r.stdout)
print('pins reproduced from repo-only data:', rep['pins_reproduced'])
print('728 root without source bytes:', rep['ln_mini_728'])

In [ ]:
dev = json.loads((R/'redteam/receipts/ROOT_REPLAY_magicSTUDIObox.json').read_text())
print('magicSTUDIObox source-byte replay:', dev['pins_reproduced'], dev['predecessor_15']['source_bytes'])
print({k: v for k, v in dev['ln_mini_728'].items() if k != 'pinned_root'})

In [ ]:
r = subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_redteam_attacks', '-v'], capture_output=True, text=True)
print(r.stderr[-2500:]); assert r.returncode == 0

In [ ]:
subprocess.run([sys.executable, 'redteam/analyze.py', 'report', '--results', 'redteam/results'], check=True)
print((R/'redteam/results/EVIDENCE_MATRIX.md').read_text())

In [ ]:
m = json.loads((R/'redteam/results/EVIDENCE_MATRIX.json').read_text())
for a in m['admission']: print(a)
assert not any(a['admitted'].startswith('ADMIT') for a in m['admission'] if a['comparison'].startswith('K'))

In [ ]:
iv = json.loads(subprocess.run([sys.executable, 'redteam/impact_view.py', '.'], capture_output=True, text=True, check=True).stdout)
print(json.dumps(iv['arm10_changed_source_version'], indent=1))